In [1]:
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
load_dotenv()

True

In [2]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="conversational",
    temperature=0.3,
    max_new_tokens=512,
)

model = ChatHuggingFace(llm=llm)

e:\LangGraph\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    rating : int

In [12]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f"Create an outline for a blog post with the title: {title}"
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [ ]:
def create_blog(state: BlogState) -> BlogState:
    outline = state['outline']
    title = state['title']
    prompt = f"Write a detailed blog post with the title: {title} and the following outline: {outline}"
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [13]:
def create_rating(state: BlogState) -> BlogState:
    outline = state['outline']
    content = state['content']
    prompt = f"Rate the quality of the blog post with the title: {state['title']} and the following outline: {outline} and content: {content}. Provide a score from 1 to 10."
    content = model.invoke(prompt).content
    state['rating'] = content
    return state


In [14]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('blog_rating',create_rating)

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','blog_rating')
graph.add_edge('blog_rating',END)

#compile
workflow = graph.compile()

In [15]:
initial_state = {'title': 'Crime Rate in India'}
final_state=workflow.invoke(initial_state) # this will return {'title': 'The Future of AI in India', 'outline': '...', 'content': '...'}
print(final_state)

{'title': 'Crime Rate in India', 'outline': 'Certainly! Below is an outline for a blog post titled "Crime Rate in India":\n\n---\n\n**Title: Crime Rate in India**\n\n**Introduction**\n- Brief overview of the topic\n- Importance of understanding crime rates in India\n- Purpose of the blog post\n\n**Section 1: Overview of Crime in India**\n- Definition and types of crimes\n- Historical context of crime in India\n- Current state of crime in India\n\n**Section 2: Statistics and Trends**\n- Recent crime statistics (e.g., from the National Crime Records Bureau, NCRB)\n- Trends over the past decade\n- Regional variations in crime rates\n\n**Section 3: Key Factors Contributing to Crime**\n- Socio-economic factors\n- Demographic changes\n- Urbanization and migration\n- Political and legal factors\n\n**Section 4: Impact of Crime on Society**\n- Psychological and emotional impact on victims\n- Economic impact on families and communities\n- Social and cultural implications\n\n**Section 5: Governme

In [16]:
print(final_state['outline'])

Certainly! Below is an outline for a blog post titled "Crime Rate in India":

---

**Title: Crime Rate in India**

**Introduction**
- Brief overview of the topic
- Importance of understanding crime rates in India
- Purpose of the blog post

**Section 1: Overview of Crime in India**
- Definition and types of crimes
- Historical context of crime in India
- Current state of crime in India

**Section 2: Statistics and Trends**
- Recent crime statistics (e.g., from the National Crime Records Bureau, NCRB)
- Trends over the past decade
- Regional variations in crime rates

**Section 3: Key Factors Contributing to Crime**
- Socio-economic factors
- Demographic changes
- Urbanization and migration
- Political and legal factors

**Section 4: Impact of Crime on Society**
- Psychological and emotional impact on victims
- Economic impact on families and communities
- Social and cultural implications

**Section 5: Government Initiatives and Responses**
- Recent government policies and programs
- Su

In [17]:
print(final_state['content'])

# Crime Rate in India

India, with its rich cultural heritage and diverse population, has a complex landscape when it comes to crime. Understanding the current state of crime in the country is crucial for both policymakers and the general public. This blog post aims to provide an overview of the crime rate in India, analyze recent trends, and discuss the factors contributing to crime, as well as the government's initiatives and the role of the community in addressing this issue.

## Section 1: Overview of Crime in India

### Definition and Types of Crimes
Crime in India encompasses a wide range of offenses, including but not limited to murder, rape, theft, robbery, and cybercrime. The National Crime Records Bureau (NCRB) categorizes these crimes into different types, each with its own set of implications for society.

### Historical Context of Crime in India
Historically, India has seen fluctuations in crime rates, influenced by various socio-economic and political factors. The colonia

In [18]:
print(final_state['rating'])

Based on the provided outline and content, I would rate the quality of the blog post as follows:

**Rating: 7/10**

### Positive Aspects:
1. **Structured Outline**: The outline is well-organized, covering various important aspects of the topic, which provides a clear roadmap for the blog post.
2. **Comprehensive Coverage**: The content covers a wide range of topics, including historical context, current state, statistics, trends, contributing factors, government initiatives, and community involvement.
3. **Use of Data and Statistics**: The inclusion of recent data from the National Crime Records Bureau (NCRB) adds credibility and supports the analysis.
4. **Engaging Introduction**: The introduction sets the stage well by providing an overview of the topic and explaining its importance.

### Areas for Improvement:
1. **Depth and Specificity**: The content could benefit from more detailed analysis and specific examples. For instance, providing more specific data points, case studies, and